In [ ]:
import numpy as np
import pandas as pd
import os
import wandb
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    classification_report
)
from sklearn.preprocessing import label_binarize
import torch
from torch.optim import AdamW
from transformers.optimization import get_linear_schedule_with_warmup
import gc

In [ ]:
os.environ["WANDB_API_KEY"] = "4ac0249f5d4c7d9239c6a0ec893195a256a55f46"
wandb.login()

In [ ]:
wandb_config = {
    "model_name": "FacebookAI/roberta-base",
    "task": "nepali_multiclass_classification",
    "learning_rate": 2e-5,  
    "batch_size": 8,
    "gradient_accumulation_steps": 2,  
    "num_epochs": 10,  
    "max_length": 256,
    "weight_decay": 0.01,
    "adam_epsilon": 1e-8, 
    "adam_beta1": 0.9,
    "adam_beta2": 0.999,
    "max_grad_norm": 1.0,  
    "train_split": 0.8,
    "val_split": 0.1,
    "test_split": 0.1,
    "random_seed": 42,
    "warmup_ratio": 0.1,  
    "eval_steps": 1000,  
    "logging_steps": 100,
    "save_steps": 1000,
    "early_stopping_patience": 30,
    "label_smoothing": 0.1, 
    "fp16": False,  
    "bf16": False,
}

In [ ]:
run = wandb.init(
    project="topic-modeling-25k-experiments",
    name="roberta-25k-2e5-256-16-10",
    config=wandb_config,
    tags=["nepali", "bert", "classification", "roberta"]
)

In [ ]:
df = pd.read_excel("/home/rupak/Desktop/Topic-Modeling /dataset/topic-modeling-25k-dataset.xlsx")
df.head()

In [ ]:
df = df.sample(frac=1, random_state=wandb.config.random_seed).reset_index(drop=True)
df.head()

In [ ]:
unique_labels = sorted(df["domainid"].unique())
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}
df["labels"] = df["domainid"].map(label2id)

In [ ]:
dataset = Dataset.from_pandas(df[['relevant_sentences', 'labels']])
dataset = dataset.rename_column('relevant_sentences', 'text')

In [ ]:
dataset.shape

In [ ]:
train_test_split = dataset.train_test_split(
    test_size=0.2,
    seed=wandb.config.random_seed,
    shuffle=True
)
train_dataset = train_test_split['train']

In [ ]:
val_test_split = train_test_split['test'].train_test_split(
    test_size=0.5,
    seed=wandb.config.random_seed,
    shuffle=True
)
val_dataset = val_test_split['train']  # 10% of total
test_dataset = val_test_split['test']   # 10% of total

In [ ]:
dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

In [ ]:
model_name = wandb.config.model_name
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding=False,  # Dynamic padding with data collator
        truncation=True,
        max_length=wandb.config.max_length
    )

In [ ]:
dataset = dataset.map(tokenize, batched=True, remove_columns=['text'])

In [ ]:
num_labels = len(unique_labels)
num_labels

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    dtype=torch.float32,   # corrected
    device_map="auto",
    trust_remote_code=True,
    local_files_only=False,  # set True if model is cached locally
    ignore_mismatched_sizes=True
)

In [ ]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt"
)

In [ ]:
total_steps = (len(dataset["train"]) // (wandb.config.batch_size * wandb.config.gradient_accumulation_steps)) * wandb.config.num_epochs
warmup_steps = int(total_steps * wandb.config.warmup_ratio)

In [ ]:
training_args = TrainingArguments(
    output_dir="/home/rupak/Desktop/Topic-Modeling /topic modeling 25k/checkpoint/roberta-25k-2e5-256-16-10",
    eval_strategy="steps",
    eval_steps=wandb.config.eval_steps,
    save_strategy="steps",
    save_steps=wandb.config.save_steps,
    learning_rate=wandb.config.learning_rate,
    per_device_train_batch_size=wandb.config.batch_size,
    per_device_eval_batch_size=wandb.config.batch_size,
    gradient_accumulation_steps=wandb.config.gradient_accumulation_steps,
    num_train_epochs=wandb.config.num_epochs,
    weight_decay=wandb.config.weight_decay,
    adam_beta1=wandb.config.adam_beta1,
    adam_beta2=wandb.config.adam_beta2,
    adam_epsilon=wandb.config.adam_epsilon,
    max_grad_norm=wandb.config.max_grad_norm,
    warmup_steps=warmup_steps,
    lr_scheduler_type="linear",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_weighted",
    greater_is_better=True,
    save_total_limit=3,
    label_smoothing_factor=wandb.config.label_smoothing,
    # Enhanced logging
    logging_dir='./logs',
    logging_steps=wandb.config.logging_steps,
    logging_first_step=True,
    # Wandb integration
    report_to="wandb",
    run_name=f"nepberta-optimized-{wandb.run.id}",
    # Full precision training (FP32)
    fp16=wandb.config.fp16,
    bf16=wandb.config.bf16,
    dataloader_drop_last=False,
    eval_accumulation_steps=None,
    # Additional optimization
    dataloader_num_workers=2,
    remove_unused_columns=False,
    push_to_hub=False,
)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # Get probabilities for AUROC
    probabilities = torch.softmax(torch.tensor(logits), dim=-1).numpy()

    # Calculate comprehensive metrics
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted', zero_division=0
    )

    # Macro averages for unbiased evaluation
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, predictions, average='macro', zero_division=0
    )

    # Calculate AUROC (for multiclass)
    if num_labels > 2:
        labels_binarized = label_binarize(labels, classes=range(num_labels))
        try:
            auroc_weighted = roc_auc_score(labels_binarized, probabilities, multi_class='ovr', average='weighted')
            auroc_macro = roc_auc_score(labels_binarized, probabilities, multi_class='ovr', average='macro')
        except ValueError:
            auroc_weighted = 0.0
            auroc_macro = 0.0
    else:
        auroc_weighted = roc_auc_score(labels, probabilities[:, 1])
        auroc_macro = auroc_weighted

    metrics = {
        "accuracy": accuracy,
        "precision_weighted": precision,
        "recall_weighted": recall,
        "f1_weighted": f1,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "auroc_weighted": auroc_weighted,
        "auroc_macro": auroc_macro
    }

    return metrics

In [ ]:
class CustomTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.train_accuracy_history = []
        self._last_train_metrics = {}



    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        """
        Compute loss and optionally calculate training accuracy
        """
        labels = inputs.get("labels")
        outputs = model(**inputs)
        loss = outputs.get("loss")

        # Calculate training accuracy periodically
        if (self.state.global_step % (self.args.logging_steps * 2) == 0 and
            labels is not None and self.state.global_step > 0):
            with torch.no_grad():
                predictions = torch.argmax(outputs.logits, dim=-1)
                accuracy = (predictions == labels).float().mean().item()
                self._last_train_metrics = {"train_accuracy": accuracy}

        return (loss, outputs) if return_outputs else loss

In [ ]:
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=wandb.config.early_stopping_patience)]
)

In [ ]:
trainer.train()

In [ ]:
test_results = trainer.evaluate(eval_dataset=dataset["test"])

In [ ]:
for key, value in test_results.items():
    if key.startswith('eval_'):
        metric_name = key.replace('eval_', '').replace('_', ' ').title()
        print(f"{metric_name:<20}: {value:.4f}")

In [ ]:
# Get detailed predictions for test set
test_predictions = trainer.predict(dataset["test"])
test_logits = test_predictions.predictions
test_labels = test_predictions.label_ids
test_pred_labels = np.argmax(test_logits, axis=-1)


In [ ]:
# Convert back to original domain labels for interpretability
test_pred_domains = [id2label[pred] for pred in test_pred_labels]
test_true_domains = [id2label[label] for label in test_labels]

In [ ]:
report = classification_report(
    test_labels, 
    test_pred_labels, 
    target_names=[id2label[i] for i in range(num_labels)],
    digits=4
)

In [ ]:
print(report)